In [ ]:
#uv add langchain langchain-community langchain-teddynote langchain-openai langchain-classic langsmith langchain-chroma pypdf pydantic

Resolved 159 packages in 2ms
Checked 152 packages in 216ms


In [6]:
from dotenv import load_dotenv
load_dotenv()
from langchain_teddynote import logging
logging.langsmith("Summary")

from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

client = Client()

map_llm = ChatOpenAI(
    temperature=0.1,
    model_name = "gpt-4o-mini",
)

map_summary = client.pull_prompt(
    "teddynote/map-summary-prompt",
    dangerously_pull_public_prompt=True
)

map_summary.pretty_print()

LangSmith 추적을 시작합니다.
[프로젝트명]
Summary
================================ System Message ================================

You are an expert summarizer. Your task is to summarize the following document in {language}.

================================ Human Message =================================

Extract most important main thesis from the documents, then summarize in bullet points.

#Format:
- summary 1
- summary 2
- summary 3
-...

Here is a given document: 
{documents}

Write 1~5 sentences. Think step by step.
#Summary:


In [7]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("data/Truth.pdf")
docs = loader.load()
docs = docs[0:9]

map_chain = map_summary | map_llm | StrOutputParser()
print(map_chain.invoke({"documents": docs[0], "language": "Korean"}))

- 매일일보의 조정훈 인턴기자가 작성한 7건의 기사 내용을 요약한 자료입니다.
- 이 자료는 RAG, VectorStore, Retriever와 같은 기술을 활용한 실습용 예제 PDF입니다.
- 주요 분야는 정치, 외교, 국정감사, 국방, 문화 논쟁 등입니다.
- 문서 수집 기준일은 2026년 9월 29일입니다.
- 원문 전체를 복제한 것이 아니라 핵심 내용을 재구성한 것입니다.


In [8]:
input_doc=[{"documents":doc, "language":"Korean"} for doc in docs]
input_doc

[{'documents': Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-29T02:32:20+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-09-29T02:32:20+00:00', 'subject': '(unspecified)', 'title': '매일일보 조정훈 인턴기자 기사 실습 자료', 'trapped': '/False', 'source': 'data/Truth.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='매일일보 조정훈 인턴기자 기사 - RAG/Retriever 실습용 요약\n1\n 매일일보 조정훈 인턴기자\n 기사 7건 실습 자료\n RAG · VectorStore · Retriever · 문서 분할 실습용\n최근 매일일보에 게재된 조정훈 인턴기자 기사 7건의 제목과 공개 검색 결과를 바탕으로 핵심 내용을 짧게 재구성한 예제 PDF입니다.\n원문 전체를 복제한 자료가 아닙니다.\n문서 수\n7건\n수집 기준일\n2026-09-29\n주요 분야\n정치 · 외교 · 국정감사 · 국방 · 문화 논쟁\n추천 실습\nText Splitter → Embedding → VectorStore → Retriever'),
  'language': 'Korean'},
 {'documents': Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-29T02:32:20+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-09-29T02:3

In [9]:
print(map_chain.batch(input_doc))

['- 매일일보의 조정훈 인턴기자가 작성한 7건의 기사 내용을 요약한 자료입니다.\n- 이 자료는 RAG, VectorStore, Retriever와 같은 기술을 활용한 실습용 예제 PDF입니다.\n- 문서의 주요 분야는 정치, 외교, 국정감사, 국방, 문화 논쟁입니다.\n- 원문 전체를 복제한 것이 아니라 핵심 내용을 재구성한 것입니다.\n- 수집 기준일은 2026년 9월 29일입니다.', '- 야당이 국정감사를 앞두고 공세를 강화하고 있으며, 장외 집회에 대해서는 신중한 입장을 보이고 있다.\n- 2026년 국정감사에서 운영위원회, 재정위원회, 행정안전위원회가 주요 쟁점으로 부각되고 있으며, 인사, 부동산, 경찰 개혁이 논의될 예정이다.\n- 미중 정상회담에서 양국 간의 웃음이 있었지만, AI, 대만, 이란 문제에 대해서는 여전히 입장 차이가 존재한다.\n- 이 대통령은 알래스카에서 귀국하는 길에 즉석 차담회를 가졌다.\n- 정점식 의원은 대통령의 무능을 비판하며 국민의 불만이 계속되고 있다고 주장했다.', '- 국민의힘이 국정감사를 앞두고 정부에 대한 공세를 강화하고 있다.\n- 당 내부에서는 장외투쟁의 필요성에 대한 공감이 있지만, 실제 집회 추진에는 신중한 태도를 보이고 있다.\n- 국정감사 대응과 장외투쟁 전략의 조합이 주요 쟁점으로 부각되고 있다.', '- 2026년도 국정감사는 10월 6일부터 27일까지 3주 동안 진행될 예정이다.\n- 운영위원회에서는 청와대 인사검증 논란이 주요 쟁점으로 다뤄진다.\n- 재정경제 관련 상임위에서는 부동산 및 세제 현안이 핵심 이슈로 제시된다.\n- 행정안전위원회에서는 경찰개혁 문제가 중요한 이슈로 부각된다.\n- 이 기사는 여러 상임위에서 다뤄질 현안을 정리하여 국정감사 전체 지형을 미리 파악할 수 있도록 구성되었다.', '- 도널드 트럼프 미국 대통령과 시진핑 중국 국가주석의 정상회담에서 양국은 무역과 인공지능 분야의 협력을 강조했다.\n- 그러나 AI, 대만, 이란과 같은 주요 현안에서는 여전히 입장차가 존

In [10]:
refine_prompt = client.pull_prompt(
    "teddynote/refine-prompt",
    dangerously_pull_public_prompt=True
)

refine_prompt.pretty_print()

================================ System Message ================================

You are an expert summarizer.

================================ Human Message =================================

Your job is to produce a final summary

We have provided an existing summary up to a certain point:
{previous_summary}

We have the opportunity to refine the existing summary(only if needed) with some more context below.
------------
{current_summary}
------------
Given the new context, refine the original summary in {language}.
If the context isn't useful, return the original summary.


In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain
from langchain_teddynote.callbacks import StreamingCallback
from langsmith import Client
from langchain_community.document_loaders import PyPDFLoader

client = Client()

loader = PyPDFLoader("data/Truth.pdf")
docs = loader.load()
docs = docs[0:9]

refine_prompt = client.pull_prompt(
    "teddynote/refine-prompt",
    dangerously_pull_public_prompt=True
)

refine_llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4o-mini",
)

refine_chain = refine_prompt | refine_llm | StrOutputParser()

from langchain_core.runnables import chain

@chain
def map_refine_chain(docs):

    map_summary = client.pull_prompt(
        "teddynote/map-summary-prompt",
        dangerously_pull_public_prompt=True
    )

    map_chain = (
        map_summary
        | ChatOpenAI(
            model_name = "gpt-4o-mini",
            temperature=0.1,
        )
        |StrOutputParser()
    )

    input_doc = [{"documents": doc.page_content, "language": "Korean"} for doc in docs]

    doc_summaries = map_chain.batch(input_doc)

    refine_prompt = client.pull_prompt(
        "teddynote/refine-prompt",
        dangerously_pull_public_prompt=True
    )

    refine_llm = ChatOpenAI(
        model_name = "gpt-4o-mini",
        temperature=0.1,
        callbacks = [StreamingCallback()],
        streaming = True,
    )

    refine_chain = refine_prompt | refine_llm | StrOutputParser()

    previous_summary = doc_summaries[0]

    for current_summary in doc_summaries[1:]:
        previous_summary = refine_chain.invoke(
            {
                "previous_summary": previous_summary,
                "current_summary": current_summary,
                "language": "Korean",
            }
        )
        print("\n\n---------------------\n\n")

    return previous_summary

refined_summary = map_refine_chain.invoke(docs)


- 매일일보의 조정훈 인턴기자가 작성한 7건의 기사 내용을 요약한 자료입니다.
- 이 자료는 RAG, VectorStore, Retriever와 같은 기술을 활용한 실습용 예제입니다.
- 주요 분야는 정치, 외교, 국정감사, 국방, 문화 논쟁 등입니다.
- 문서 수집 기준일은 2026년 9월 29일입니다.
- 원문 전체를 복제한 것이 아니라 핵심 내용을 재구성한 것입니다.
- 국감이 다가오면서 야당의 공세가 강화되고 있으며, 장외 집회에 대한 신중한 접근이 필요하다는 의견이 제기되고 있습니다. 2026년 국감에서는 운영위원회, 재경위원회, 행정안전위원회의 주요 쟁점으로 인사, 부동산, 경찰 개혁이 예상됩니다. 미중 정상회담에서는 양국 간의 웃음이 있었으나, AI, 대만, 이란 문제에 대해서는 여전히 입장 차이가 존재합니다. 이 대통령은 알래스카에서 귀국 중 즉석 차담회를 가졌으며, 정점식 의원은 현 정부를 무능하다고 비판하며 국민의 불만이 계속되고 있다고 언급했습니다.

---------------------


- 매일일보의 조정훈 인턴기자가 작성한 7건의 기사 내용을 요약한 자료입니다.
- 이 자료는 RAG, VectorStore, Retriever와 같은 기술을 활용한 실습용 예제입니다.
- 주요 분야는 정치, 외교, 국정감사, 국방, 문화 논쟁 등입니다.
- 문서 수집 기준일은 2026년 9월 29일입니다.
- 원문 전체를 복제한 것이 아니라 핵심 내용을 재구성한 것입니다.
- 국민의힘이 국정감사를 앞두고 정부에 대한 공세를 강화하고 있으며, 당 내부에서는 장외투쟁의 필요성에 대한 공감이 있지만 실제 집회 추진에는 신중한 태도를 보이고 있습니다. 국정감사에서는 운영위원회, 재경위원회, 행정안전위원회의 주요 쟁점으로 인사, 부동산, 경찰 개혁이 예상됩니다. 미중 정상회담에서는 양국 간의 웃음이 있었으나, AI, 대만, 이란 문제에 대해서는 여전히 입장 차이가 존재합니다. 이 대통령은 알래스카에서 귀국 중 즉석 차담회를 가졌으며, 정점식 의원은 현 정부를 